In [1]:
import pandas as pd

# Load the uploaded Excel file to check its structure
file_path = '../dataset/rab94row.xlsx'
data = pd.read_excel(file_path)

# Display the first few rows to understand the structure
data.head()

,namaproyek,rab,waktu,provinsi,tahun,luas,subitem,tinggi,lantai,ikk,ihbp,inflasi
0,Gedung Apartemen,871000000000,606,DKI Jakarta,2021,68204.25309,6,120.020872,31,121.42,109.64,1.53
1,Gedung Apartemen,876000000000,605,DKI Jakarta,2021,68590.48909,6,120.617753,31,121.42,109.64,1.53
2,Gedung Apartemen,936000000000,649,DKI Jakarta,2020,73232.95902,6,128.564830,33,116.84,103.68,1.59
3,Gedung Apartemen,938000000000,628,DKI Jakarta,2020,73318.17099,6,128.882451,33,116.84,103.68,1.59
4,Gedung Apartemen,1050000000000,668,DKI Jakarta,2020,81640.56895,6,142.594757,36,116.84,103.68,1.59


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, Lasso, LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [3]:
# Label encoding kolom 'Provinsi' dan nama proyek karena nilainya kategorikal
label_encoder = LabelEncoder()
# data['label_provinsi'] = label_encoder.fit_transform(data['provinsi'])
data['label_namaproyek'] = label_encoder.fit_transform(data['namaproyek'])

# Melihat nilai unik dari hasil encoding kolom 'provinsi_encoded'
# provinsi_unik = data[['provinsi', 'label_provinsi']].drop_duplicates().sort_values(
#     by='label_provinsi')

# Melihat nilai unik dari hasil encoding kolom 'namaproyek_encoded'
namaproyek_unik = data[['namaproyek', 'label_namaproyek']].drop_duplicates().sort_values(
    by='label_namaproyek')

In [4]:
# 6. One-Hot Encoding untuk 'provinsi' dan 'namaproyek'
data = pd.get_dummies(data, columns=['provinsi'])

In [5]:
data.head()

,namaproyek,rab,waktu,tahun,luas,subitem,tinggi,lantai,ikk,ihbp,inflasi,label_namaproyek,provinsi_Bali,provinsi_Banten,provinsi_DKI Jakarta,provinsi_Jawa Barat,provinsi_Jawa Timur,provinsi_Kalimantan Timur,provinsi_Sulawesi Selatan
0,Gedung Apartemen,871000000000,606,2021,68204.25309,6,120.020872,31,121.42,109.64,1.53,0,False,False,True,False,False,False,False
1,Gedung Apartemen,876000000000,605,2021,68590.48909,6,120.617753,31,121.42,109.64,1.53,0,False,False,True,False,False,False,False
2,Gedung Apartemen,936000000000,649,2020,73232.95902,6,128.564830,33,116.84,103.68,1.59,0,False,False,True,False,False,False,False
3,Gedung Apartemen,938000000000,628,2020,73318.17099,6,128.882451,33,116.84,103.68,1.59,0,False,False,True,False,False,False,False
4,Gedung Apartemen,1050000000000,668,2020,81640.56895,6,142.594757,36,116.84,103.68,1.59,0,False,False,True,False,False,False,False


In [6]:
# Split data into features and target
X1 = data.drop(columns=['rab', 'namaproyek'])
y1 = data['rab']

# Split into training and testing sets
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X1, y1, 
    test_size=0.3, 
    random_state=42)

# Split into training and testing sets
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X1, y1, 
    test_size=0.2, 
    random_state=42)



In [10]:
# Initialize models
models = {
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'Bayesian Ridge': BayesianRidge(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'KNN Regressor': KNeighborsRegressor(),
    'Lasso Regression': Lasso(random_state=42),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Ridge Regression': Ridge(random_state=42),
    'SVR': SVR(),
    'XGBoost': xgb.XGBRegressor(random_state=42)
}

# 70:30
# Train and evaluate each model with additional metrics
results1 = []
for name1, model1 in models.items():
    # Fit model to training data
    model1.fit(X_train1, y_train1)
    
    # Predict on training data
    y_train_pred1 = model1.predict(X_train1)
    
    # Predict on testing data
    y_pred1 = model1.predict(X_test1)
    
    # Calculate metrics for training data
    r2_train1 = r2_score(y_train1, y_train_pred1)
    
    # Calculate metrics for testing data
    mse1 = mean_squared_error(y_test1, y_pred1)
    mae1 = mean_absolute_error(y_test1, y_pred1)
    r21 = r2_score(y_test1, y_pred1)
    
    # Append results
    results1.append({
        'Model': name1,
        'Training R2 Score': r2_train1,
        'Testing RMSE': np.sqrt(mse1),
        'Testing MAE': mae1,
        'Testing R2 Score': r21
    })

# Convert results into a DataFrame for better visualization
results_df1 = pd.DataFrame(results1).sort_values(by='Testing R2 Score', ascending=False)

# 80:20
# Train and evaluate each model with additional metrics
results2 = []
for name2, model2 in models.items():
    # Fit model to training data
    model2.fit(X_train2, y_train2)
    
    # Predict on training data
    y_train_pred2 = model2.predict(X_train2)
    
    # Predict on testing data
    y_pred2 = model2.predict(X_test2)
    
    # Calculate metrics for training data
    r2_train2 = r2_score(y_train2, y_train_pred2)
    
    # Calculate metrics for testing data
    mse2 = mean_squared_error(y_test2, y_pred2)
    mae2 = mean_absolute_error(y_test2, y_pred2)
    r22 = r2_score(y_test2, y_pred2)
    
    # Append results
    results2.append({
        'Model': name2,
        'Training R2 Score': r2_train2,
        'Testing RMSE': np.sqrt(mse2),
        'Testing MAE': mae2,
        'Testing R2 Score': r22
    })

# Convert results into a DataFrame for better visualization
results_df2 = pd.DataFrame(results2).sort_values(by='Testing R2 Score', ascending=False)


Hasil 70:30

In [11]:
results_df1

,Model,Training R2 Score,Testing RMSE,Testing MAE,Testing R2 Score
5,Lasso Regression,9.998445e-01,1.969740e+09,1.359104e+09,0.999858
6,Linear Regression,9.998446e-01,1.970574e+09,1.375896e+09,0.999858
8,Ridge Regression,9.998412e-01,1.973194e+09,1.336603e+09,0.999857
7,Random Forest,9.985094e-01,1.625937e+10,7.806552e+09,0.990323
4,KNN Regressor,9.756463e-01,1.625962e+10,7.489655e+09,0.990323
3,Gradient Boosting,9.999999e-01,1.832660e+10,7.629258e+09,0.987706
0,AdaBoost,9.975573e-01,2.123270e+10,1.128539e+10,0.983498
2,Decision Tree,1.000000e+00,2.982709e+10,1.117241e+10,0.967436
10,XGBoost,1.000000e+00,3.421999e+10,1.415981e+10,0.957138
1,Bayesian Ridge,7.280843e-13,1.663792e+11,1.362345e+11,-0.013236


Hasil 80:20

In [12]:
results_df2

,Model,Training R2 Score,Testing RMSE,Testing MAE,Testing R2 Score
8,Ridge Regression,9.998442e-01,1.704914e+09,1.162288e+09,0.999879
6,Linear Regression,9.998481e-01,1.748197e+09,1.200824e+09,0.999873
5,Lasso Regression,9.998481e-01,1.757497e+09,1.203511e+09,0.999872
4,KNN Regressor,9.899213e-01,1.739171e+10,7.284211e+09,0.987438
7,Random Forest,9.975421e-01,2.077016e+10,1.051000e+10,0.982083
3,Gradient Boosting,9.999999e-01,2.491040e+10,9.036022e+09,0.974228
0,AdaBoost,9.967610e-01,2.597922e+10,1.478997e+10,0.971969
10,XGBoost,1.000000e+00,2.617642e+10,1.193818e+10,0.971542
2,Decision Tree,1.000000e+00,3.545865e+10,1.205263e+10,0.947780
9,SVR,-2.023078e-01,1.569663e+11,1.063158e+11,-0.023297
